# Prédiction de l'Âge des Arbres

## Objectif
Prédire l'âge estimé des arbres (age_estim) à partir de leurs caractéristiques en utilisant le machine learning supervisé.

## Plan
1. Chargement des données
2. Nettoyage et sélection des variables
3. Prétraitement (encodage et normalisation)
4. Modélisation avec GridSearchCV
5. Évaluation et sauvegarde du modèle

In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

## 1. Chargement des données

In [3]:
df = pd.read_csv("Data_Arbre_Clean.csv")
print(f"Dataset: {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(df.head())

Dataset: 11248 lignes, 37 colonnes
              X             Y  OBJECTID            created_date  \
0  1.720320e+06  8.294619e+06         1  2017/02/02 15:27:50+00   
1  1.720898e+06  8.293531e+06         2  2017/02/02 15:27:50+00   
2  1.720894e+06  8.293542e+06         3  2017/02/02 15:27:50+00   
3  1.720902e+06  8.293545e+06         4  2017/02/02 15:27:50+00   
4  1.721089e+06  8.293619e+06         5  2017/02/02 15:27:50+00   

      created_user     src_geo              clc_quartier          clc_secteur  \
0  mickael.delaere  Orthophoto  Quartier du Centre-Ville  Boulevard Richelieu   
1  mickael.delaere  Orthophoto  Quartier du Centre-Ville  Boulevard Léon Blum   
2  mickael.delaere  Orthophoto  Quartier du Centre-Ville  Boulevard Léon Blum   
3  mickael.delaere  Orthophoto  Quartier du Centre-Ville  Boulevard Léon Blum   
4  mickael.delaere  Orthophoto  Quartier du Centre-Ville  Boulevard Léon Blum   

  id_arbre  haut_tot  ...  villeca  nomfrancais nomlatin  \
0       24     

## 2. Nettoyage et sélection des variables

In [4]:
df = df.dropna(subset=["age_estim"])
y = df["age_estim"]

cols_to_drop = ["OBJECTID", "id_arbre", "GlobalID", "CreationDate", "created_date", 
                "created_user", "last_edited_user", "last_edited_date", "Creator", 
                "Editor", "EditDate", "commentaire_environnement", "dte_plantation", "dte_abattage"]

X = df.drop(columns=cols_to_drop + ["age_estim"], errors="ignore")

print(f"Nombre de lignes: {len(X)}")
print(f"Nombre de features: {X.shape[1]}")
print(f"Variables categorielles: {X.select_dtypes(include=['object']).shape[1]}")
print(f"Variables numeriques: {X.select_dtypes(exclude=['object']).shape[1]}")

Nombre de lignes: 10415
Nombre de features: 22
Variables categorielles: 15
Variables numeriques: 7


C:\Users\mathe\AppData\Local\Temp\ipykernel_1628\1454515177.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(f"Variables categorielles: {X.select_dtypes(include=['object']).shape[1]}")


## 3. Séparation des variables

In [5]:
cat_columns = X.select_dtypes(include=["object"]).columns.tolist()
num_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

print(f"Colonnes categorielles: {len(cat_columns)}")
print(f"Colonnes numeriques: {len(num_columns)}")

Colonnes categorielles: 15
Colonnes numeriques: 7


C:\Users\mathe\AppData\Local\Temp\ipykernel_1628\3951226963.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns = X.select_dtypes(include=["object"]).columns.tolist()


## 4. Prétraitement

In [6]:
cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("cat", cat_transformer, cat_columns),
    ("num", num_transformer, num_columns)
])

print("Preprocessor cree")

Preprocessor cree


## 5. Separation train/test et modele

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(random_state=42)
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model)
])

print(f"Train set: {X_train.shape[0]} lignes")
print(f"Test set: {X_test.shape[0]} lignes")

Train set: 8332 lignes
Test set: 2083 lignes


## 6. Optimisation avec GridSearchCV

In [8]:
param_grid = {
    "model__n_estimators": [50, 100, 150],
    "model__max_depth": [10, 15, 20]
}

grid = GridSearchCV(pipeline, param_grid, cv=3, scoring="r2", n_jobs=-1)
grid.fit(X_train, y_train)

print(f"Meilleur score (CV): {grid.best_score_:.4f}")
print(f"Meilleurs parametres: {grid.best_params_}")

Meilleur score (CV): 0.9633
Meilleurs parametres: {'model__max_depth': 20, 'model__n_estimators': 150}


## 7. Evaluation

In [9]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"R2 score: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

R2 score: 0.9703
MAE: 1.4323
RMSE: 3.5032


## 8. Sauvegarde du modele

In [10]:
joblib.dump(best_model, "modele_arbre.pkl")
joblib.dump(preprocessor, "preprocessor_arbre.pkl")
print("Modele et preprocessor sauvegardes")

Modele et preprocessor sauvegardes
